# 02E — Four-forest leave-one-site-out transfer benchmark

**One selected method only**, read from `02D_selected_strategy.json` after the main within-site 3-fold model comparison completes. Train four models: withhold Baego, MOS (all three mosaics), Yangambi, or Yoko in turn; train on all remaining forest groups' partial annotations and exhaustive circles, without any labels from the target group. All 22 exhaustive circles are evaluated across their corresponding held-out forest test.

**Objective:** new-forest transferability, **not** optimizing the six existing orthomosaics. Full-circle fraction metrics primary, discrimination secondary. The V2 E1 illumination reference median is fit on only the three training forest groups each LOSO iteration; image-only luminance statistics at the target are permitted unsupervised and reported. No field plot use, no target reference calibration.

Checkpoints/scores: `Final/Outputs/V3/02E_LOSO/`. We intentionally do not overwrite 02C/02D outputs.


## 01 — Environment and load selected strategy

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn
from pathlib import Path
import json, warnings, math, random, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
CV_OUT=ROOT/'Outputs/V3/02D_spatial_cv'
BASE_OUT=ROOT/'Outputs/V3/02E_LOSO';BASE_OUT.mkdir(parents=True,exist_ok=True)
OUT=BASE_OUT
REF=ROOT/'Outputs/V3/01_reference_data'
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
mosaics={m['file']:m for m in A['mosaics']}
splits=pd.read_csv(REF/'01_cv_window_manifest.csv')
links=pd.read_csv(REF/'01_circle_spatial_folds.csv')
repairs=pd.read_csv(REF/'01_geometry_repairs.csv')
FOLD=0;SEED=20261016;CROP=256;EPOCHS=25;BATCH=8
TARGET_SITES=['Baego','MOS','Yangambi','Yoko']
SELECTED=json.loads((CV_OUT/'02D_selected_strategy.json').read_text())
SELECTED_MODEL=SELECTED['model'];SELECTED_TREATMENT=SELECTED['illumination']
assert SELECTED_MODEL in ['M1','M2']
assert SELECTED_TREATMENT in ['native','augment','v2_E1_site']
print('LOCKED STRATEGY',SELECTED_MODEL,SELECTED_TREATMENT)
MAX_PARTIAL_PER_MOSAIC=250
MAX_EXHAUSTIVE_PER_SITE=260
HOLDOUT_COUNTS={'Baego':1,'MOS':2,'Yangambi':2,'Yoko':2}
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU for the pilot.'
device='cuda';torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
print('GPU',torch.cuda.get_device_name(0),'| fold',FOLD)
print('Train/val candidates:',splits[splits.cv_fold==FOLD].split.value_counts().to_dict())
print('Files:',ROOT,'| output:',OUT)





## 02 — Load masks and U-Net architecture

In [ ]:
def pieces(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in pieces(q)]
    return []
def polygons(path):
    src=gpd.read_file(path);inds=[];shapes=[]
    for i,g in enumerate(src.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            try:
                fixed=unary_union(pieces(make_valid(g)))
                change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if change>.05 or not fixed.is_valid:continue
                g=fixed
            except Exception:continue
        inds.append(i);shapes.append(g)
    x=src.iloc[inds].copy().reset_index(drop=True)
    x.geometry=shapes
    x['original_row']=inds
    return x
roi=polygons(A['roi'])
circles=polygons(A['exhaustive_annotations']['complete_roi_boundaries'])
positive=polygons(A['exhaustive_annotations']['positive_polygons'])
circle_ids=set(links.circle_id.astype(int))
circles=circles[circles.original_row.isin(circle_ids)].copy()
tree={}
for site,path in A['shared_annotations'].items():
    t=polygons(path)
    cls=pd.to_numeric(t.Class_ID,errors='coerce')
    t['class01']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1))
    tree[site]=t
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as src:
        c=circles.to_crs(src.crs).copy()
        c['fold']=c.original_row.map(links.set_index('circle_id').fold)
        c['owner']=c.original_row.map(links.set_index('circle_id').mosaic)
        geo[name]=dict(site=m['site'],path=m['path'],transform=src.transform,
            crs=src.crs,roi=roi.to_crs(src.crs),
            circles=c,positive=positive.to_crs(src.crs),trees=tree[m['site']].to_crs(src.crs))



In [ ]:
class TinyUNet(nn.Module):
    def __init__(self,base=16):
        super().__init__()
        def block(a,b):
            return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
        self.e1=block(3,base);self.e2=block(base,base*2)
        self.bridge=block(base*2,base*4)
        self.d2=block(base*4+base*2,base*2)
        self.d1=block(base*2+base,base)
        self.final=nn.Conv2d(base,1,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(F.max_pool2d(e1,2))
        b=self.bridge(F.max_pool2d(e2,2))
        x=self.d2(torch.cat([F.interpolate(b,size=e2.shape[-2:],mode='bilinear',align_corners=False),e2],1))
        x=self.d1(torch.cat([F.interpolate(x,size=e1.shape[-2:],mode='bilinear',align_corners=False),e1],1))
        return self.final(x)[:,0]
class CropData(Dataset):
    def __init__(self,rows,augment=False):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        d=self.rows[i];image=d['rgb'];y=d['y'];w=d['w']
        if self.augment:
            rng=np.random.default_rng(SEED+i+int(torch.randint(0,1000000,(1,)).item()))
            k=int(rng.integers(4));image=np.rot90(image,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if rng.random()<.5:image=image[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
        return x,torch.tensor(np.ascontiguousarray(y)),torch.tensor(np.ascontiguousarray(w))
def masked_loss(logits,y,w):
    loss=F.binary_cross_entropy_with_logits(logits,y,reduction='none')
    return (loss*w).sum()/w.sum().clamp_min(1.)
print('Tiny U-Net ready (native 256×256 RGB crops).')




## 03 — LOSO reference masks: exclude all target-forest annotations

In [ ]:
from rasterio.windows import Window
from torch.utils.data import Dataset,DataLoader
candidate=pd.read_csv(REF/'01_candidate_windows.csv')
all_circle_buffers={}
BLOCKED={}
DEV=set();HOLDOUT=set()
for name,d in geo.items():
    cs=circles.to_crs(d['crs'])
    all_circle_buffers[name]=[g.buffer(15.) for g in cs.geometry]
    d['dev_circles']=cs[cs.original_row.isin(DEV)]
    d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
    d['positive_local']=positive.to_crs(d['crs'])
    d['trees_local']=tree[d['site']].to_crs(d['crs')]
    d['roi_local']=roi.to_crs(d['crs'])

def burn_window(series,name,win):
    # GeoPandas spatial index avoids scanning every tree polygon for every crop.
    # rasterize clips geometry to the requested window automatically.
    d=geo[name]
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    ids=series.sindex.query(bb,predicate='intersects')
    if len(ids)==0:
        return np.zeros((int(win.height),int(win.width)),dtype=bool)
    gs=series.iloc[ids]
    return rasterize(((geom,1) for geom in gs if geom is not None and not geom.is_empty),
       out_shape=(int(win.height),int(win.width)),
       transform=rasterio.windows.transform(win,d['transform']),
       fill=0,dtype='uint8').astype(bool)

def rgb_for_window(name,win):
    with rasterio.open(geo[name]['path']) as ds:
        raw=ds.read([1,2,3],window=win,masked=True)
    valid=~np.any(np.ma.getmaskarray(raw),axis=0)
    a=raw.filled(0).astype('float32')
    valid &= np.any(a!=0,axis=0)&np.any(a!=255,axis=0)
    return np.clip(a/255.,0,1).transpose(1,2,0).astype('float32'),valid

def training_crop(rec,model):
    name=rec.mosaic;d=geo[name]
    if rec.height<CROP or rec.width<CROP:return None
    rr=int(rec.row)+(int(rec.height)-CROP)//2
    cc=int(rec.col)+(int(rec.width)-CROP)//2
    win=Window(cc,rr,CROP,CROP)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    if any(bb.intersects(g) for g in BLOCKED[name]):return None
    if model=='M1' and any(bb.intersects(g) for g in all_circle_buffers[name]):return None
    if model=='M2' and rec.kind=='exhaustive' and int(rec.circle_id) not in DEV:return None
    inside=burn_window(d['roi_local'].geometry,name,win)
    t=d['trees_local']
    pos_tree=burn_window(t.loc[t.class01==1].geometry,name,win)
    neg_tree=burn_window(t.loc[t.class01==0].geometry,name,win)
    if model=='M1':
        full=np.zeros_like(inside);pos_ex=np.zeros_like(inside)
    else:
        full=burn_window(d['dev_circles'].geometry,name,win)&inside
        pos_ex=burn_window(d['positive_local'].geometry,name,win)&full
    conflict=(pos_tree&neg_tree)| (pos_ex&neg_tree) | (full&~pos_ex&pos_tree)
    target=(pos_ex|(~full&pos_tree)).astype('float32')
    weight=(full|((pos_tree|neg_tree)&~full))&inside&~conflict
    img,valid=rgb_for_window(name,win)
    weight &= valid
    if int(weight.sum())<250:return None
    return dict(image=img,target=target,weight=weight.astype('float32'),
                site=d['site'],kind=rec.kind,mosaic=name)

def make_training_cache(model):
    selected=[]
    for (site,kind),grp in candidate.groupby(['site','kind']):
        if model=='M1' and kind=='exhaustive':continue
        if model=='M2' and kind=='exhaustive':
            grp=grp[grp.circle_id.isin(DEV)]
        limit=MAX_EXHAUSTIVE_PER_SITE if kind=='exhaustive' else MAX_PARTIAL_PER_MOSAIC*(3 if site=='MOS' else 1)
        subset=grp.sample(n=min(limit,len(grp)),random_state=SEED)
        selected.extend(subset.itertuples(index=False))
    result=[]
    prep_started=time.perf_counter()
    for i,rec in enumerate(selected):
        chip=training_crop(rec,model)
        if chip is not None:result.append(chip)
        if (i+1)%100==0:
            print(f"{model}: prepared {i+1}/{len(selected)} candidates, usable {len(result)}, elapsed {time.perf_counter()-prep_started:.1f}s",flush=True)
    site_counts = pd.Series([item["site"] for item in result]).value_counts().to_dict()
    print(f"{model}: usable training chips = {len(result)}", flush=True)
    print("Training chips by site:", site_counts, flush=True)
    print(f"{model}: cache preparation took {time.perf_counter()-prep_started:.1f}s",flush=True)
    assert len(result)>15,f'{model}: too few chips'
    return result

class CropDataset(Dataset):
    def __init__(self,rows):self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        q=self.rows[i];x=q['image'].copy();y=q['target'].copy();w=q['weight'].copy()
        k=int(torch.randint(0,4,(1,)).item())
        x=np.rot90(x,k);y=np.rot90(y,k);w=np.rot90(w,k)
        if torch.rand(1).item()<.5:x=x[:,::-1];y=y[:,::-1];w=w[:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2,0,1))),
                torch.from_numpy(np.ascontiguousarray(y)),torch.from_numpy(np.ascontiguousarray(w)))



CANDIDATES_FULL=candidate.copy()
def configure_loso(held_site):
    global FOLD,DEV,HOLDOUT,BLOCKED,split_df,candidate,TARGET_SITE
    TARGET_SITE=held_site
    FOLD=-1
    HOLDOUT=set(links.loc[links.site==held_site,'circle_id'].astype(int))
    DEV=set(links.circle_id.astype(int))-HOLDOUT
    candidate=CANDIDATES_FULL[CANDIDATES_FULL.site!=held_site].copy()
    split_df=links[['circle_id','site','mosaic','cover_pct']].copy()
    split_df['split']=np.where(split_df.circle_id.isin(HOLDOUT),'holdout','development')
    BLOCKED={}
    for name,d in geo.items():
        cs=circles.to_crs(d['crs'])
        d['dev_circles']=cs[cs.original_row.isin(DEV)]
        d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
        held_here=set(links[(links.site==d['site'])&links.circle_id.isin(HOLDOUT)].circle_id.astype(int))
        BLOCKED[name]=[g.buffer(15.) for g in cs[cs.original_row.isin(held_here)].geometry]
    print('LOSO target',held_site,'target circles',sorted(HOLDOUT),
          'training candidate sites',candidate.site.unique().tolist(),flush=True)
    assert held_site not in candidate.site.unique()
    assert set(candidate.site.unique())==set(['Baego','MOS','Yangambi','Yoko'])-{held_site}


## 04 — E1 gains with strictly training-site reference, or native/augmentation treatment

In [ ]:
from rasterio.windows import Window
from scipy.stats import linregress
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
RGB_TREATMENT='native'
_ORIG_RGB=rgb_for_window
def luminance(x):
    return .2126*x[:,:,0]+.7152*x[:,:,1]+.0722*x[:,:,2]
def sample_mosaic_luminance(name,n=80,side=256):
    rng=np.random.default_rng(SEED+sum(ord(q) for q in name))
    values=[]
    d=geo[name]
    with rasterio.open(d['path']) as src:
        for i in range(n):
            # Stratified, seeded, spread over image extent; exclude nodata and nearly empty patches
            fraction=(i+.5)/n
            row=int(np.clip(((i%9)+rng.random())/9*max(0,src.height-side),0,max(0,src.height-side)))
            col=int(np.clip(((i//9)+rng.random())/max(1,int(np.ceil(n/9)))*max(0,src.width-side),
                            0,max(0,src.width-side)))
            rgb=src.read([1,2,3],window=Window(col,row,side,side))
            a=np.moveaxis(rgb,0,-1).astype('float32')/255.
            valid=~((rgb==0).all(axis=0)|(rgb==255).all(axis=0))
            if valid.mean()<.8:continue
            values.append(float(np.median(luminance(a)[valid])))
    if len(values)<3:
        # Sparse valid footprints: use already-audited candidate-window locations
        # as an image-only sampling fallback. No reference labels are read.
        fallback=candidate[candidate.mosaic==name]
        with rasterio.open(d['path']) as src:
            for row in fallback.sample(n=min(60,len(fallback)),random_state=SEED).itertuples(index=False):
                rr=int(row.row);cc=int(row.col)
                h=min(side,src.height-rr);w=min(side,src.width-cc)
                if h<=0 or w<=0:continue
                rgb=src.read([1,2,3],window=Window(cc,rr,w,h))
                a=np.moveaxis(rgb,0,-1).astype('float32')/255.
                valid=~((rgb==0).all(axis=0)|(rgb==255).all(axis=0))
                if valid.mean()>=.8:values.append(float(np.median(luminance(a)[valid])))
                if len(values)>=10:break
    if len(values)<3:
        raise RuntimeError(f'{name}: insufficient valid RGB even after annotated-window fallback ({len(values)})')
    return float(np.median(values)),len(values)
MOSAIC_LUM={name:sample_mosaic_luminance(name) for name in geo}
SITE_LUM={}
for site in ['Baego','MOS','Yangambi','Yoko']:
    values=[v for name,(v,n) in MOSAIC_LUM.items() if geo[name]['site']==site]
    SITE_LUM[site]=float(np.median(values))
SITE_GAIN={}
def configure_site_illumination(held_site):
    global SITE_GAIN,REF_LUM
    other=[s for s in SITE_LUM if s!=held_site]
    REF_LUM=float(np.median([SITE_LUM[s] for s in other]))
    SITE_GAIN={s:float(np.clip(REF_LUM/SITE_LUM[s],.85,1.15)) for s in SITE_LUM}
    print('LOSO reference luminance (TRAINING SITES ONLY):',REF_LUM,
          '| target unsupervised gain',SITE_GAIN[held_site],flush=True)
    return dict(heldout_site=held_site,training_sites=other,
                site_gains=SITE_GAIN,train_reference=REF_LUM)
def rgb_for_window(name,win):
    rgb,valid=_ORIG_RGB(name,win)
    if RGB_TREATMENT=='v2_E1_site':
        rgb=np.clip(rgb*SITE_GAIN[geo[name]['site']],0,1).astype('float32')
    return rgb,valid


## 05 — Selected model training implementation

In [ ]:
class IlluminationCropDataset(CropDataset):
    def __init__(self,rows,mode):
        super().__init__(rows);self.mode=mode
    def __getitem__(self,i):
        image,y,w=super().__getitem__(i)
        if self.mode=='augment':
            b=float(torch.empty(1).uniform_(.80,1.20))
            contrast=float(torch.empty(1).uniform_(.85,1.15))
            gamma=float(torch.empty(1).uniform_(.8,1.2))
            image=torch.clamp((image-.5)*contrast+.5,0,1)
            image=torch.clamp(image*b,0,1)**gamma
        return image,y,w
import time
def fit_network(label):
    ckpt=OUT/f'{label}_model.pt'
    torch.manual_seed(SEED)
    model=TinyUNet().to(device)
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt,map_location=device,weights_only=True))
        print('Loaded',ckpt,flush=True)
        return model
    print(f"{label}: building training cache...",flush=True)
    cache=make_training_cache(label)
    print(f"{label}: starting GPU training, {EPOCHS} epochs, {len(cache)} chips",flush=True)
    loader=DataLoader(IlluminationCropDataset(cache,RGB_TREATMENT),batch_size=BATCH,shuffle=True,num_workers=0)
    optim=torch.optim.AdamW(model.parameters(),lr=7e-4,weight_decay=1e-4)
    history=[]
    for epoch in range(EPOCHS):
        model.train();losses=[];start=time.time()
        for x,y,w in loader:
            x=x.to(device);y=y.to(device);w=w.to(device)
            optim.zero_grad(set_to_none=True)
            loss=masked_loss(model(x),y,w)
            loss.backward();optim.step()
            losses.append(float(loss.item()))
        history.append(dict(epoch=epoch+1,masked_BCE=float(np.mean(losses))))
        pd.DataFrame(history).to_csv(OUT/f'{label}_loss.csv',index=False)
        print(f'{label}: epoch {epoch+1}/{EPOCHS} BCE={np.mean(losses):.4f}, {time.time()-start:.1f}s',flush=True)
    torch.save(model.state_dict(),ckpt)
    del cache
    return model




## 06 — Full-circle LOSO evaluation

In [ ]:
from scipy.special import expit,logit
from scipy.optimize import minimize
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
WINDOWS=pd.read_csv(REF/'01_candidate_windows.csv')
VAL_WINDOWS=WINDOWS[WINDOWS.kind=='exhaustive'].copy()
assert set(VAL_WINDOWS.circle_id.unique().astype(int))==set(links.circle_id.astype(int))
N_BINS=256
BINS=np.linspace(0,1,N_BINS+1)
BIN_CENTER=(BINS[:-1]+BINS[1:])/2

def run_all_circles(model,model_name):
    statfile=OUT/f'{model_name}_circle_raw.csv'
    histfile=OUT/f'{model_name}_histograms.npz'
    if statfile.exists() and histfile.exists():
        return pd.read_csv(statfile),dict(np.load(histfile))
    model.eval();rows=[];histograms={}
    for _,circle in split_df[split_df.split=='holdout'].iterrows():
        cid=int(circle.circle_id)
        name=circle.mosaic
        d=geo[name]
        windows=VAL_WINDOWS[(VAL_WINDOWS.circle_id==cid)&(VAL_WINDOWS.mosaic==name)]
        assert len(windows),f'No evaluation windows for circle {cid}'
        cg=circles[circles.original_row==cid].to_crs(d['crs'])
        counts=np.zeros(N_BINS,np.int64);true_counts=np.zeros(N_BINS,np.int64)
        n=0;sumy=0.;sump=0.;sum_sq=0.;tp=fp=fn=0
        rng=np.random.default_rng(SEED+cid)
        sample_truth=[];sample_probs=[]
        for rec in windows.itertuples(index=False):
            win=Window(int(rec.col),int(rec.row),int(rec.width),int(rec.height))
            mask=burn_window(cg.geometry,name,win)
            mask &= burn_window(d['roi_local'].geometry,name,win)
            rgb,valid=rgb_for_window(name,win)
            mask &= valid
            if not mask.any():continue
            reference=burn_window(d['positive_local'].geometry,name,win)
            x=torch.from_numpy(rgb.transpose(2,0,1).copy())[None].to(device)
            with torch.inference_mode():
                pred=torch.sigmoid(model(x))[0].cpu().numpy()
            y=reference[mask].astype('uint8');p=pred[mask].astype('float32')
            n+=len(y);sumy+=float(y.sum());sump+=float(p.sum())
            sum_sq+=float(np.square(p-y).sum())
            binary=p>=.5
            tp+=int(np.sum(binary&(y==1)));fp+=int(np.sum(binary&(y==0)))
            fn+=int(np.sum((~binary)&(y==1)))
            ix=np.minimum((p*N_BINS).astype(np.int32),N_BINS-1)
            counts+=np.bincount(ix,minlength=N_BINS).astype('int64')
            true_counts+=np.bincount(ix,weights=y,minlength=N_BINS).astype('int64')
            # only diagnostic AUC/AP uses pixel subsampling, with fixed per-circle cap
            take=rng.choice(len(y),min(100,len(y)),replace=False)
            sample_truth.extend(y[take].tolist());sample_probs.extend(p[take].tolist())
        assert n>0,f'No valid ROI/RGB reference pixels for circle {cid}'
        assert int(counts.sum())==n
        yt=np.asarray(sample_truth);pr=np.asarray(sample_probs)
        rows.append(dict(model=model_name,circle_id=cid,site=circle.site,split=circle.split,
                         valid_pixels=n,observed_fraction=100*sumy/n,
                         predicted_fraction=100*sump/n,
                         bias_pp=100*(sump-sumy)/n,brier=sum_sq/n,
                         f1_threshold_05=2*tp/max(1,2*tp+fp+fn),
                         auc_sample=roc_auc_score(yt,pr) if len(np.unique(yt))>1 else np.nan,
                         ap_sample=average_precision_score(yt,pr) if len(np.unique(yt))>1 else np.nan))
        histograms[str(cid)+'_count']=counts
        histograms[str(cid)+'_positive']=true_counts
        print(model_name,'circle',cid,'obs',round(100*sumy/n,1),
              'pred',round(100*sump/n,1),'valid pixels',n,flush=True)
    frame=pd.DataFrame(rows)
    assert len(frame)==len(HOLDOUT)
    frame.to_csv(statfile,index=False)
    np.savez_compressed(histfile,**histograms)
    return frame,histograms



## 07 — Run/resume four strict site holdouts

In [ ]:
import gc
RGB_TREATMENT=SELECTED_TREATMENT
for site in TARGET_SITES:
    configure_loso(site)
    OUT=BASE_OUT/site
    OUT.mkdir(parents=True,exist_ok=True)
    gains=configure_site_illumination(site)
    (OUT/'02E_illumination_contract.json').write_text(json.dumps(gains,indent=2))
    score_file=OUT/f'{SELECTED_MODEL}_circle_raw.csv'
    checkpoint=OUT/f'{SELECTED_MODEL}_model.pt'
    if score_file.exists() and checkpoint.exists():
        print('SKIP completed LOSO:',site,flush=True)
        continue
    model=fit_network(SELECTED_MODEL)
    result,_=run_all_circles(model,SELECTED_MODEL)
    assert result.circle_id.nunique()==len(HOLDOUT),(site,len(result),len(HOLDOUT))
    assert set(result.site)=={site}
    print('LOSO completed:',site,'circles',len(result),'fraction MAE pp',
          round(float(result.bias_pp.abs().mean()),3),flush=True)
    del model;torch.cuda.empty_cache();gc.collect()
print('Four-site LOSO runs completed or resumed.')


## 08 — Four-site transferability summary and observed/predicted plot

In [ ]:
frames=[]
for site in TARGET_SITES:
    file=BASE_OUT/site/f'{SELECTED_MODEL}_circle_raw.csv'
    assert file.exists(),f'Run Section 07 first: {file}'
    frames.append(pd.read_csv(file))
LOSO=pd.concat(frames,ignore_index=True)
assert LOSO.circle_id.nunique()==22
def metrics(q):
    y=q.observed_fraction.to_numpy(float);p=q.predicted_fraction.to_numpy(float)
    r=linregress(y,p) if len(q)>=3 and len(np.unique(y))>1 else None
    return pd.Series(dict(n_circles=len(q),R2=r2_score(y,p) if len(q)>=3 else np.nan,
        MAE_pp=mean_absolute_error(y,p),RMSE_pp=np.sqrt(mean_squared_error(y,p)),
        mean_bias_pp=float(np.mean(p-y)),slope=r.slope if r else np.nan,
        intercept_pp=r.intercept if r else np.nan,brier=q.brier.mean(),
        AUC=q.auc_sample.mean(),AP=q.ap_sample.mean()))
BY_SITE=LOSO.groupby('site').apply(metrics,include_groups=False).reset_index()
OVERALL=metrics(LOSO).to_frame().T
display(BY_SITE.round(3));display(OVERALL.round(3))
LOSO.to_csv(BASE_OUT/'02E_all_site_holdouts.csv',index=False)
BY_SITE.to_csv(BASE_OUT/'02E_site_metrics.csv',index=False)
OVERALL.to_csv(BASE_OUT/'02E_overall_metrics.csv',index=False)
fig,ax=plt.subplots(figsize=(7,7))
ax.plot([0,100],[0,100],ls='--',color='gray')
colors=dict(zip(TARGET_SITES,['#c27e24','#357ebd','#468c50','#944b99']))
for site,g in LOSO.groupby('site'):
    ax.scatter(g.observed_fraction,g.predicted_fraction,label=site,color=colors[site],s=46)
ax.set(xlim=(0,100),ylim=(0,100),xlabel='Observed canopy fraction (%)',
       ylabel='Predicted canopy fraction (%)',
       title='Four-site LOSO: entirely unseen forest groups')
ax.legend();ax.grid(alpha=.2)
fig.savefig(BASE_OUT/'02E_LOSO_observed_predicted.png',dpi=180)
display(fig);plt.close(fig)
print('This is transferability evidence; choose the production approach using 02D spatial CV.')
